In [ ]:
import re
import pandas as pd
import csv
import os

import glob
# load ground truth files
DRUG_ID_MAP = {
    "upadacitinib": "DB15091",
    "digitoxin": "DB01396",
    "simvastatin": "DB00641",
}

def read_pharma_gt_csv(x):
    gt = pd.read_csv(x)
    dataset = x.split("/")[1]
    gt["DRUG_A"] = dataset
    gt["DRUG_A_ID"] = DRUG_ID_MAP[dataset]
    gt["GT"] = gt.GT.apply(lambda x: x.lower())
    return gt

gt_csvs = glob.glob("phase1/**/*final_dataset.csv", recursive=False)


pharma_gt = pd.concat([read_pharma_gt_csv(x) for x in gt_csvs])

ID_TO_DRUGNAME_MAP = {row["DRUG_B_ID"]:row["DRUG_B_NAME"] for _, row in pharma_gt.iterrows()}
for k,v in DRUG_ID_MAP.items():
    ID_TO_DRUGNAME_MAP[v] = k.capitalize()

# Direct Prompt

In [ ]:
P_X = """Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??)."""

# Few Shot Exemplars

In [ ]:
P_E5 = """Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??).
Below you find extra context to help you answer the question.

(A) Cefepime (B) Probenecid => When probenecid is used to elevate plasma concentrations of penicillin or other beta-lactams, or when such drugs are given to patients taking probenecid, high plasma concentrations of the other drug may increase the incidence of adverse reactions associated with that drug. In the case of penicillin or other beta-lactam administration, such as cephalosporins, psychiatric disturbances have been observed. ?? YES ??
(A) Mavorixafor (B) Tramadol => Tramadol is a substrate of CYP2D6 and its concomitant use with inhibitors of these enzymes may result in an increase in the serum concentration of tramadol and a decrease in the serum concentration of its main and active metabolite, M1.2 As M1 is a more potent agonist of opioid receptors than its parent drug, decreases in its serum concentration may impair analgesic efficacy and may induce withdrawal in patients that have developed a dependency on tramadol.2,1 In contrast, increased serum concentrations of the parent drug may increase the incidence and/or severity of adverse events such as serotonin syndrome or seizures. ?? YES ??
(A) Testosterone enanthate (B) Citalopram => No clinically relevant interaction known. ?? NO ??
(A) Eravacycline (B) Dicloxacillin => Concomitant administration of tetracycline and penicillin antibiotics may diminish the therapeutic effects of penicillins. Penicillins require actively dividing bacteria to inhibit cell wall synthesis and tetracyclines interfere with this process by inhibiting bacterial replication. There are in vitro studies that both support and challenge this theory. For pneumococcal meningitis treatment, some studies have demonstrated that penicillin monotherapy results in a decreased rate of mortality when compared to a regimen that includes both tetracycline and penicillin. In addition, investigations have shown that rates of H. pylori eradication were substantially lower when amoxicillin and tetracycline were administered together compared to control arm therapies. Other studies have shown that successive use of the antibiotics can be useful in some circumstances. For example: a course of tetracycline following standard treatment for gonorrhea can effectively erradicate chlamydia for those who are co-infected. ?? YES ??YES
(A) Opicapone (B) Aripiprazole lauroxil => Antipsychotics are often used in the treatment of psychotic symptoms associated with Parkinson's disease. The atypical antipsychotic agents are known to cause parkinson-like adverse effects via the blockade of dopamine receptors. This can decrease the therapeutic efficacy of anti-parkinson drugs, which are used to control such symptoms. In addition to the above effects, additive sedative effects may also occur. ?? YES ??

Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??)."""

P_E10 = """Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??).
Below you find extra context to help you answer the question.

(A) Damoctocog alfa pegol (B) Dermatan sulfate => Blood coagulation factors promote the blood coagulation pathways to ultimately form the insoluble fibrin clot. In contrast, fibrinolytic agents activate the fibrinolytic system by conversion of the inactive proenzyme, plasminogen into the active enzyme plasmin, that degrades fibrin to break down the insoluble clot. Desired procoagulant effects of blood coagulation factors may be reduced with the combination use of fibrinolytic agents. ?? YES ?? 

(A) Magnesium oxide (B) Delafloxacin => Fluoroquinolone antibiotics, such as the affected drug, have a propensity to form chelate complexes with cations, such as magnesium. Chelation results in impaired absorption of the antibiotic in question, reducing serum concentration and, therefore, antibiotic efficacy. Common sources of cations (including magnesium) are antacids, multivitamins, and other nutritional supplements. ?? YES ?? 

(A) Conivaptan (B) Pimozide => The subject drug is a strong CYP3A4 enzyme inhibitor, and the affected drug is metabolized by the CYP3A4 enzyme. Concomitant administration of these agents will decrease the metabolism of the CYP3A4 substrate, increasing its serum concentration and therapeutic effect. Drugs with a narrow therapeutic index must be maintained within a specific concentration range in order to be safe and efficacious. An increased concentration of a drug with a narrow therapeutic index can lead to significant adverse effects and toxicity. ?? YES ?? 

(A) Dienogest (B) Sodium citrate => Oral contraceptives may increase blood clotting factors and decrease the effectiveness of anticoagulants due to their prothrombotic effects.1,2,4 In contrast, in some patients anticoagulant effects may actually be potentiated by the use of oral contraceptives, although this risk appears to be relatively low. ?? YES ?? 

(A) Asparaginase Erwinia chrysanthemi (B) Cinchocaine => The use of local anesthetics has been associated with the development of methemoglobinemia, a rare but serious and potentially fatal adverse effect. The concurrent use of local anesthetics and oxidizing agents such as antineoplastic agents may increase the risk of developing methemoglobinemia. ?? YES ?? 

(A) Secretin human (B) Flavoxate => Pancreatic secretion has been shown to have cholinergic input. The ability of HumanSecretin to stimulate pancreatic secretion of bicarbonate, water, and proteins may be reduced by concomitant use of anticholinergic agents. Both antimuscarinic and antinicotinic agents have been shown to produce a reduction in secretion. ?? YES ?? 

(A) Thiamphenicol (B) Procaine => The use of local anesthetics has been associated with the development of methemoglobinemia, a rare but serious and potentially fatal adverse effect. The concurrent use of local anesthetics and oxidizing agents such as antibiotics may increase the risk of developing methemoglobinemia. ?? YES ?? 

(A) Zopiclone (B) Cinolazepam => The co-administration of ethanol with zopiclone increased the risk of adverse effects such as complex sleep behaviors (sleep driving, eating food, making phone calls, leaving the house), and also increases the CNS depressant effects of zopiclone. This may result in profound sedation or respiratory depression. ?? YES ??
(A) Griseofulvin (B) Vinblastine => The subject drug is a CYP3A4 enzyme inducer of unknown strength, and the affected drug is metabolized by the CYP3A4 enzyme. Concomitant administration of these agents will induce the metabolism of the CYP3A4 substrate (affected drug), reducing the serum concentration and therapeutic effect. Drugs with a narrow therapeutic index must be maintained within a specific concentration range in order to be safe and efficacious. Reduced concentration of a drug with a narrow therapeutic index may lead to significantly lower efficacy. ?? YES ??
(A) Armodafinil (B) Pantoprazole => Modafinil and its isomer, armodafinil, were shown to reversibly inhibit CYP2C19 in vitro Co-administration of modafinill or armodafinil with drugs that are cleared by CYP2C19-mediated metabolism may result in increased serum concentrations and higher systemic exposure of the substrates due to the inhibition of CYP2C19. In clinical studies, such interaction was seen with some drugs that were subject to CYP2C19 metabolism, such as omeprazole. ?? YES ??

Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??)."""

# Mechanism Prompt

In [ ]:
P_M = """Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??).
Below you find extra context to help you answer the question.

# 1 Build a structured profile for each drug
- Disposition: dominant vs secondary clearance mechanisms; relevant transport proteins; substrate, inhibitor, inducer, or activator roles
- Absorption: dependencies or liabilities that co-administered agents could alter (dissolution environment, complexation, transit)
- Pharmacodynamics: primary mechanisms and direction of effect on physiological systems
- Fragility factors: therapeutic window breadth, reliance on bioactivation or a single pathway, persistence, steep exposure-response, active-metabolite dependence

# 2 PK interaction appraisal (High, Moderate, Low)
- Pathway interference: one drug alters a pathway or transporter that is dominant for the other drug's clearance → High; partial or secondary involvement → Moderate; non-overlapping or negligible → Low
- Capacity competition: both depend on the same capacity-limited step in elimination or uptake → elevate by one level
- Absorption interference: co-administration plausibly alters absorption conditions or binding/complexation → assign a level as above

# 3 PD interaction appraisal (High, Moderate, Low)
- Additive or synergistic effects on safety-relevant physiological endpoints → High
- Opposing effects likely to cause instability of a safety-relevant endpoint → High
- Overlap with uncertain clinical consequence → Moderate
- Minimal overlap or unrelated systems → Low

# 4 Elevation by risk modifiers
- If any fragility factor applies to either drug, elevate the higher of PK or PD by one level (Low→Moderate, Moderate→High)

# 5 Decision rule
- YES if PK High or PD High (after elevation)
- YES if both PK Moderate and PD Moderate are present
- YES if two distinct Moderate mechanisms are present (for example, PK Moderate plus absorption Moderate)
- Otherwise NO

Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??)."""

# Hybrid Prompt

In [ ]:
P_H = """Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??).
Below you find extra context to help you answer the question.

Example for (A) Mavorixafor (B) Vardenafil
# 1) Assemble knowledge
## Mavorixafor
- Enzymes (metabolism): Primarily metabolized by CYP3A4 (major route).
- Transporters: No major role from intestinal/hepatic/renal transporters identified.
- Perpetrator profile: CYP3A4 inhibitor (at least moderate potency reported in clinical/pharmacology data).
- PD liabilities: No major overlapping toxicity with PDE5 inhibitors.
- NTI: No.

## Vardenafil
- Enzymes (metabolism): Primarily eliminated by hepatic cytochrome metabolism via CYP3A4 (major route), with minor CYP3A5/CYP2C isoform contributions.
- Transporters: No clinically critical transporter dependence for clearance.
- Perpetrator profile: Not a significant enzyme inhibitor or inducer at therapeutic levels.
- PD liabilities: PDE5 inhibition → vasodilation, hypotension risk, especially if exposure increases.
- NTI: No.

# 2) Enzyme-mediated PK interaction
- Perpetrator: Mavorixafor inhibits CYP3A4.
- Victim: Vardenafil is primarily cleared by CYP3A4.
- Rule 2a applies → Inhibition of major clearance pathway → clinically relevant ↑ exposure expected.

# 3) Transporter-mediated PK interaction
- Not applicable — no critical transporters involved.

# 4) Absorption-level interaction
- No evidence for GI pH or chelation effects.

# 5) Pharmacodynamic interaction
- Both can cause mild vasodilation but no high-risk PD overlap beyond PK-exposure increase.

# 6) Authoritative signals
- No “contraindicated/avoid” labeling known internally, but general PDE5 inhibitor caution with strong/moderate CYP3A4 inhibitors is consistent.

# Decision
YES → Condition 2a satisfied.
Mechanistic basis: Mavorixafor inhibits CYP3A4, the main clearance enzyme for vardenafil, leading to higher plasma concentrations.
?? YES ??

Does a clinically relevant drug‑drug‑interaction exist between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (e.g., ?? YES ?? or ?? NO ??)."""

# AI Mechanism Prompt

In [ ]:
P_MAI = """### **Guideline for Determining Clinically Relevant Drug Interactions**

A clinically relevant interaction occurs when the co-administration of two drugs leads to a significant change in the therapeutic effect or adverse effect profile of one or both drugs. To determine this, check for the following two major types of interactions: Pharmacodynamic and Pharmacokinetic.

If the answer to **any** of the questions below is "yes," the interaction is clinically relevant.

---

#### **Part 1: Pharmacodynamic Interactions (What the drugs do to the body)**

This type of interaction involves drugs acting on the same or related physiological systems.

*   **Check for Antagonism:** Do the two drugs have opposing effects?
    *   **Question:** Will one drug's mechanism of action directly counteract the desired effect of the other drug?
*   **Check for Additive or Synergistic Effects:** Do the two drugs have similar effects that combine?
    *   **Question:** Do both drugs share a significant therapeutic or adverse effect, such that their combined use would dangerously enhance this effect?

---

#### **Part 2: Pharmacokinetic Interactions (What the body does to the drugs)**

This type of interaction involves one drug affecting the Absorption, Distribution, Metabolism, or Excretion (ADME) of another.

*   **Check for Absorption Issues:** Does one drug prevent the other from being absorbed into the bloodstream?
    *   **Question:** Does one drug physically or chemically prevent the other from being absorbed?
*   **Check for Metabolism Issues (Most Common):** Does one drug alter the enzyme system (most often Cytochrome P450, or CYP enzymes) that metabolizes the other?
    *   **Inhibition:** Drug A is an **inhibitor** of an enzyme that breaks down Drug B. This will cause levels of Drug B to **increase**, leading to a higher risk of **toxicity and adverse effects**. This is especially dangerous if Drug B has a **narrow therapeutic index** (the gap between a therapeutic dose and a toxic dose is small).
    *   **Induction:** Drug A is an **inducer** of an enzyme that breaks down Drug B. This will cause levels of Drug B to **decrease**, leading to a loss of effectiveness and potential **treatment failure**. This is also critical for drugs with a narrow therapeutic index.
    *   **Question:** Is one drug a known inhibitor or inducer of a key metabolic pathway for the other drug, leading to a significant change in its concentration?

---

### **Final Decision Framework**

1.  Analyze the drug pair based on the Pharmacodynamic and Pharmacokinetic checkpoints above.
2.  If **any** of the checks result in a predictable, significant negative outcome (including reduced efficacy, treatment failure, increased toxicity, or an increased risk of a serious adverse event), the interaction is clinically relevant. End your response with ?? YES ??.
3.  If no such interaction is identified, it is not considered clinically relevant. In this case, end your response with ?? NO ??.

---

Now, please apply this guideline to the question whether there exists a clinically relevant drug‑drug‑interaction between (A) {a} and (B) {b}?
At the end of your response, output your final answer as a binary YES or NO wrapped in double question marks (?? YES ?? or ?? NO ??)."""

# Constructing the files
***

In [ ]:
prompt_types = dict(P_X=P_X, P_E5=P_E5, P_E10=P_E10, P_H=P_H, P_MAI=P_MAI, P_M=P_M)

In [ ]:
def create_prompt_dicts(row):
    dicts = []
    try:
        for prompt_type, prompt_template in prompt_types.items():
            p = prompt_template.format(a=row.DRUG_A.capitalize(), b=row.DRUG_B_NAME)
            cid = row.DRUG_A_ID + "__" + row.DRUG_B_ID + "__" + prompt_type + "__0"
            d = dict(custom_id=cid, method="POST",url="/v1/chat/completions", 
                     body=dict(model="X", messages=[dict(role="user",content=p)]))
            dicts.append(d)
    except:
        print(row)
        raise
    return dicts



In [ ]:
g = pharma_gt.groupby("DRUG_A")
for drug_a in g.groups.keys():
    requests = g.get_group(drug_a).apply(create_prompt_dicts , axis=1).explode()
    requests.to_json(f"data/{drug_a}.jsonl", orient="records", lines=True, force_ascii=False)
